<a href="https://colab.research.google.com/github/rimshidali/HuggingFace/blob/main/Abstractive_Summarization_on_News_Articles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install  datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 27.6 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import datasets
print(datasets.__version__)


3.6.0


In [2]:
from datasets import load_dataset

dataset = load_dataset("EdinburghNLP/xsum")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

xsum.py: 0.00B [00:00, ?B/s]

0000.parquet:   0%|          | 0.00/304M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/17.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})

In [ ]:
dataset["train"][0]

{'document': 'The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.\nRepair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.\nTrains on the west coast mainline face disruption due to damage at the Lamington Viaduct.\nMany businesses and householders were affected by flooding in Newton Stewart after the River Cree overflowed into the town.\nFirst Minister Nicola Sturgeon visited the area to inspect the damage.\nThe waters breached a retaining wall, flooding many commercial properties on Victoria Street - the main shopping thoroughfare.\nJeanette Tate, who owns the Cinnamon Cafe which was badly affected, said she could not fault the multi-agency response once the flood hit.\nHowever, she said more preventative work could have been carried out to ensure the retaining wall did not fail.\n"It is difficult but I do think there is so much publicity for Dumfries and the Nith - and I totally apprecia

In [3]:
# Check for null or empty values in document/summary
def check_inconsistencies(split):
    null_docs = sum(doc is None or str(doc).strip() == "" for doc in split["document"])
    null_summaries = sum(summ is None or str(summ).strip() == "" for summ in split["summary"])
    short_summaries = sum(len(str(summ).split()) < 3 for summ in split["summary"])

    return {
        "null_documents": null_docs,
        "null_summaries": null_summaries,
        "short_summaries(<3 words)": short_summaries
    }

print("Train inconsistencies:", check_inconsistencies(dataset["train"]))
print("Validation inconsistencies:", check_inconsistencies(dataset["validation"]))
print("Test inconsistencies:", check_inconsistencies(dataset["test"]))

Train inconsistencies: {'null_documents': 28, 'null_summaries': 0, 'short_summaries(<3 words)': 51}
Validation inconsistencies: {'null_documents': 5, 'null_summaries': 0, 'short_summaries(<3 words)': 2}
Test inconsistencies: {'null_documents': 1, 'null_summaries': 0, 'short_summaries(<3 words)': 1}


In [8]:
import torch
from transformers import AutoTokenizer


# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

# Tokenization function
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["document"],
        max_length=512,
        truncation=True
    )

    # Tokenize targets (labels)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["summary"],
            max_length=128,
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize all splits
tokenized_datasets = dataset.map(preprocess_function, batched=True)

print(tokenized_datasets)


Using device: cuda


Map:   0%|          | 0/204045 [00:00<?, ? examples/s]

 12%|█▏        | 25092/204045 [31:49<3:46:54, 13.14it/s]


Map:   0%|          | 0/11332 [00:00<?, ? examples/s]

Map:   0%|          | 0/11334 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11334
    })
})


In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq

import os
os.environ["WANDB_DISABLED"] = "true"


# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer and models
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
raw_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(device)  # for later comparison
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(device)      # for fine-tuning

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Minimal Training arguments for older versions
training_args = TrainingArguments(
    output_dir="./bart_xsum_finetuned",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),  # use mixed precision if GPU supports
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(2000)),  # subset for speed
    eval_dataset=tokenized_datasets["validation"].shuffle(seed=42).select(range(500)),
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Fine-tune
trainer.train()


Using device: cuda


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-516542464.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,2.957100


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3909: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=500, training_loss=2.957106689453125, metrics={'train_runtime': 319.982, 'train_samples_per_second': 6.25, 'train_steps_per_second': 1.563, 'total_flos': 2089351259258880.0, 'train_loss': 2.957106689453125, 'epoch': 1.0})

In [15]:
# Save fine-tuned model in a variable for later comparison
finetuned_model = model

In [16]:
from transformers import pipeline

# Create summarization pipelines
raw_summarizer = pipeline("summarization", model=raw_model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)
finetuned_summarizer = pipeline("summarization", model=finetuned_model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Pick a sample from validation set
sample_text = dataset["validation"][0]["document"]
reference_summary = dataset["validation"][0]["summary"]

# Generate summaries
raw_output = raw_summarizer(sample_text, max_length=128, min_length=30, do_sample=False)
finetuned_output = finetuned_summarizer(sample_text, max_length=128, min_length=30, do_sample=False)

print("\nOriginal Document:\n", sample_text)
print("\nReference Summary:\n", reference_summary)
print("\nRaw Model Summary:\n", raw_output[0]["summary_text"])
print("\nFine-tuned Model Summary:\n", finetuned_output[0]["summary_text"])


Device set to use cuda:0
Device set to use cuda:0
Your max_length is set to 128, but your input_length is only 123. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)
Your max_length is set to 128, but your input_length is only 123. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)



Original Document:
 The ex-Reading defender denied fraudulent trading charges relating to the Sodje Sports Foundation - a charity to raise money for Nigerian sport.
Mr Sodje, 37, is jointly charged with elder brothers Efe, 44, Bright, 50 and Stephen, 42.
Appearing at the Old Bailey earlier, all four denied the offence.
The charge relates to offences which allegedly took place between 2008 and 2014.
Sam, from Kent, Efe and Bright, of Greater Manchester, and Stephen, from Bexley, are due to stand trial in July.
They were all released on bail.

Reference Summary:
 Former Premier League footballer Sam Sodje has appeared in court alongside three brothers accused of charity fraud.

Raw Model Summary:
 Sam Sodje, 37, is jointly charged with elder brothers Efe, 44, Bright, 50 and Stephen, 42. The charge relates to offences which allegedly took place between 2008 and 2014. The ex-Reading defender denied fraudulent trading charges.

Fine-tuned Model Summary:
 A man has been arrested on suspicio

In [19]:
!pip install evaluate

In [21]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=636470a9594303b4f94c1bddf3e451ce287fe1d963efea87922d692412938041
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [22]:

import evaluate

rouge = evaluate.load("rouge")

def evaluate_model(model, tokenizer, dataset, num_samples=200):
    inputs = dataset.select(range(num_samples))["document"]
    refs = dataset.select(range(num_samples))["summary"]

    summaries = []
    for doc in inputs:
        tokens = tokenizer(doc, return_tensors="pt", truncation=True, max_length=512).to(device)
        output_tokens = model.generate(**tokens, max_length=128, min_length=30)
        summary = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        summaries.append(summary)

    results = rouge.compute(predictions=summaries, references=refs)
    return results

print("Raw model ROUGE:", evaluate_model(raw_model, tokenizer, dataset["validation"]))
print("Fine-tuned model ROUGE:", evaluate_model(finetuned_model, tokenizer, dataset["validation"]))


Raw model ROUGE: {'rouge1': np.float64(0.20327475560287028), 'rouge2': np.float64(0.03918437224553373), 'rougeL': np.float64(0.1376274200972549), 'rougeLsum': np.float64(0.13746438284620638)}
Fine-tuned model ROUGE: {'rouge1': np.float64(0.1660033843100643), 'rouge2': np.float64(0.02328597100673333), 'rougeL': np.float64(0.13781741657016303), 'rougeLsum': np.float64(0.1381363659634268)}
